<a href="https://colab.research.google.com/github/speediedan/finetuning-scheduler/blob/main/src/fts_examples/notebooks/publish/tunable_views/cross_architecture_addresses.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

# Addressing Across Architectures: Portable Addresses and Schedules

This notebook demonstrates the **cross-architecture portability** of our **address and schedule abstractions**. One
set of addresses, and the schedules written with them, resolve on **four model families with divergent
nomenclature**. An experiment written this way names what it trains by role and
architecture-neutral component (`role:adapter & component:mlp.*`), not by any one model's parameter names. Its
definition therefore depends far less on the model it first ran on: the same conditions, phases and sets can be run,
compared and orchestrated across models (by hand, by a sweep, or by an agent) without per-model edits, as long as each
model has a view.

The [LoRA placement notebook](lora_placement.ipynb) writes its experimental conditions as such addresses. Here those
same conditions, unchanged, are compiled against Qwen3, Llama (SmolLM2), OPT and GPT-2. Nothing is trained and no
weights are downloaded: each model is built on PyTorch's meta device from its configuration alone, so its parameters
have shapes but no storage, and the whole notebook runs in seconds on a CPU.

It shows three things:

1. **one address, four sets of names**: `component:mlp.out & layers:0` is `mlp.down_proj` on Qwen3 and Llama, `fc2`
   on OPT and `mlp.c_proj` on GPT-2, and the address is the same on all four;
2. **an address that does not apply is refused**: GPT-2 computes its queries, keys and values with one fused
   projection, so `component:attn.q` matches nothing there, and compiling a schedule that names it fails, naming the
   address, rather than quietly training nothing;
3. **one set of schedules, four models**: adapters are placed by address on all four families (the same line for
   each), and the LoRA notebook's five condition schedules compile on every family, with their tensors and parameter
   counts compared side by side.

Like the rest of the tunable views API, addresses and component maps are experimental and may change.

## Setup

The families compared, by Hugging Face model id: only each model's `config.json` is downloaded.

In [2]:
MODELS = {
    "Qwen3": "Qwen/Qwen3-0.6B",
    "Llama": "HuggingFaceTB/SmolLM2-135M",
    "OPT": "facebook/opt-125m",
    "GPT-2": "openai-community/gpt2",
}
RANK = 16

In [3]:
import warnings

import torch
from transformers import AutoConfig, AutoModelForCausalLM

from finetuning_scheduler.core.compile import compile_schedule
from finetuning_scheduler.core.exceptions import FTSConfigurationError
from venngram.maps import MappedView, PEFT_LORA_RULES

from fts_examples.tunable_views.lora import add_lora

warnings.filterwarnings("ignore", message=".*addresses.*experimental")  # said once per process

## Four Models, No Weights

`AutoModelForCausalLM.from_config` under `torch.device("meta")` builds the module tree and allocates nothing.
`MappedView` then finds each model's component map by its architecture, among the maps venngram ships and any
registered in the process, and reports every tensor's layer and architecture-neutral component.

In [4]:
def build(model_id):
    with torch.device("meta"):
        return AutoModelForCausalLM.from_config(AutoConfig.from_pretrained(model_id))


models = {family: build(model_id) for family, model_id in MODELS.items()}
for family, model in models.items():
    view = MappedView(model)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{family:<6} {type(model).__name__:<18} map {view.cmap.architecture:<18} "
          f"{n_params / 1e6:6.1f}M parameters, {view.structure().depth()} layers")

Qwen3  Qwen3ForCausalLM   map Qwen3ForCausalLM    596.0M parameters, 28 layers
Llama  LlamaForCausalLM   map LlamaForCausalLM    134.5M parameters, 30 layers
OPT    OPTForCausalLM     map OPTForCausalLM      125.2M parameters, 12 layers
GPT-2  GPT2LMHeadModel    map GPT2LMHeadModel     124.4M parameters, 12 layers


## One Address, Four Sets of Names

A schedule's `params` entry is resolved through the view when the schedule is compiled. `resolve` below compiles a
one-phase schedule and returns the tensors it thaws, or the reason it was refused.

In [5]:
def resolve(address, view):
    try:
        return compile_schedule({0: {"params": [address]}}, view).phases[0].physical
    except FTSConfigurationError as err:
        return f"refused: {str(err).rsplit(': ', 1)[-1]}"


for address in ("component:mlp.out & layers:0", "component:ln2 & layers:0", "component:attn.q & layers:0",
                "component:attn.qkv & layers:0"):
    print(address)
    for family, model in models.items():
        print(f"  {family:<6} {resolve(address, MappedView(model))}")

component:mlp.out & layers:0
  Qwen3  ('model.layers.0.mlp.down_proj.weight',)
  Llama  ('model.layers.0.mlp.down_proj.weight',)
  OPT    ('model.decoder.layers.0.fc2.weight', 'model.decoder.layers.0.fc2.bias')
  GPT-2  ('transformer.h.0.mlp.c_proj.weight', 'transformer.h.0.mlp.c_proj.bias')
component:ln2 & layers:0
  Qwen3  ('model.layers.0.post_attention_layernorm.weight',)
  Llama  ('model.layers.0.post_attention_layernorm.weight',)
  OPT    ('model.decoder.layers.0.final_layer_norm.weight', 'model.decoder.layers.0.final_layer_norm.bias')
  GPT-2  ('transformer.h.0.ln_2.weight', 'transformer.h.0.ln_2.bias')
component:attn.q & layers:0
  Qwen3  ('model.layers.0.self_attn.q_proj.weight',)
  Llama  ('model.layers.0.self_attn.q_proj.weight',)
  OPT    ('model.decoder.layers.0.self_attn.q_proj.weight', 'model.decoder.layers.0.self_attn.q_proj.bias')
  GPT-2  refused: it matches no tensor
component:attn.qkv & layers:0
  Qwen3  refused: it matches no tensor
  Llama  refused: it matches no 

The first two addresses name each family's own modules, biases included where the family has them. The last two
show the refusal: Qwen3, Llama and OPT have separate query, key and value projections (`attn.q`, `attn.k`, `attn.v`),
GPT-2 has one fused projection (`attn.qkv`), and each address resolves only where its component exists. A shared
address book can therefore say what it needs (`component:attn.q`) and find out at compile time, per model, whether a
model can provide it.

Patterns cover both: `component:attn.*` is every attention component on any of the four.

## Placing Adapters by Address

The LoRA placement notebook names its adapter targets by attribute (`q_proj`, ..., `down_proj`), which is specific to
Qwen3's module names. Here the targets are chosen by address instead: the owners of every two-dimensional attention
and MLP weight, `component:attn.* | component:mlp.*`, which leaves out Qwen3's per-head query and key norms (also
attention components, but one-dimensional). The same line places adapters on all four families, GPT-2's `Conv1D`
projections included.

In [6]:
adapted = {}
for family, model_id in MODELS.items():
    model = build(model_id)
    params = dict(model.named_parameters())
    weights = resolve("component:attn.* | component:mlp.*", MappedView(model))
    targets = sorted({name.rsplit(".", 1)[0] for name in weights if name.endswith(".weight") and params[name].dim() == 2})
    add_lora(model, r=RANK, modules=targets)
    adapted[family] = model
    print(f"{family:<6} {len(targets):>3} projections adapted, e.g. {targets[0]}")

Qwen3  196 projections adapted, e.g. model.layers.0.mlp.down_proj
Llama  210 projections adapted, e.g. model.layers.0.mlp.down_proj


OPT     72 projections adapted, e.g. model.decoder.layers.0.fc1


GPT-2   48 projections adapted, e.g. transformer.h.0.attn.c_attn


## The Conditions, Side by Side

The LoRA placement notebook's five conditions, unchanged, compiled against each family's view (with PEFT's naming
rules, so every LoRA tensor is an `adapter` at the component of the projection it adapts). The table gives each
condition's trainable parameters per family, in millions, and its share of the adapted model's parameters.

In [7]:
CONDITIONS = {
    "attention only": {0: {"params": ["role:adapter & component:attn.*"]}},
    "MLP only": {0: {"params": ["role:adapter & component:mlp.*"]}},
    "all layers": {0: {"params": ["role:adapter"]}},
    "MLP + norms": {0: {"params": ["role:adapter & component:mlp.*", "role:base & (component:ln1 | component:ln2)"]}},
    "MLP, then attention": {0: {"params": ["role:adapter & component:mlp.*"], "max_transition_step": 150},
                            1: {"params": ["role:adapter & component:attn.*"]}},
}
print(f"{'condition':<20}" + "".join(f"{family:>16}" for family in adapted))
for condition, schedule in CONDITIONS.items():
    row = f"{condition:<20}"
    for family, model in adapted.items():
        params = dict(model.named_parameters())
        compiled = compile_schedule(schedule, MappedView(model, role_rules=PEFT_LORA_RULES))
        trainable = sum(params[name].numel() for phase in compiled.phases for name in phase.physical)
        share = trainable / sum(p.numel() for p in params.values())
        row += f"{trainable / 1e6:>9.2f}M {share:>5.1%}"
    print(row)

condition                      Qwen3           Llama             OPT           GPT-2
attention only           4.59M  0.8%     1.84M  1.3%     1.18M  0.9%     0.88M  0.7%


MLP only                 5.51M  0.9%     3.04M  2.2%     1.47M  1.2%     1.47M  1.2%


all layers              10.09M  1.7%     4.88M  3.5%     2.65M  2.1%     2.36M  1.9%


MLP + norms              5.56M  0.9%     3.08M  2.2%     1.51M  1.2%     1.51M  1.2%


MLP, then attention     10.09M  1.7%     4.88M  3.5%     2.65M  2.1%     2.36M  1.9%


## What This Shows

Every condition compiles on every family, and each selects that family's own tensors: one set of schedules, four
module trees. The counts differ because the models do, and the table shows where that matters before any training:

- **The matched-parameter comparison holds on Qwen3 only.** The LoRA placement study leans on attention-only and
  MLP-only LoRA training nearly the same number of parameters at equal rank. On Qwen3 they are close (4.6M against
  5.5M), but MLP-only trains about 1.7 times as many as attention-only on GPT-2 and Llama, and 1.25 times on OPT. Run
  on those families, the comparison would need the attention-only rank raised to match.
- **The norms are a small addition everywhere.** "MLP + norms" adds every block's two normalization weights, about
  0.03M to 0.06M parameters on each family; OPT's and GPT-2's norms also have biases, Qwen3's and Llama's do not.

That is what makes an experiment defined by addresses largely model independent: its definition (the conditions, their
phases, the named sets they train) is written once, and what changes from one model to the next is only the view.
Orchestrating it across models is then a loop rather than a rewrite: for each model and each condition, compile the
schedule against the model's view, read the refusals and the counts, and only then spend compute on training. A sweep
or an agent can run that loop as readily as a person, since every model is described in the same vocabulary of roles,
layers and components.

## Using This With Your Own Models

- A model whose architecture venngram has a map for needs nothing more than `MappedView(model)`. For one it does not,
  Fine-Tuning Scheduler's `tunable-view-generator` agent skill drafts the map from the model's module tree, verifies
  it against the architecture with `venngram.maps.verify`, and registers it.
- Addresses that several schedules share belong in an address book (a small YAML document of named sets, which a
  schedule refers to by name). A book is the portable part of an experiment's definition: share it with the schedules
  that use it, and each model it runs on needs only a view. A book can require the view's naming scheme, so it is
  refused, by name, through a view with another vocabulary.
- How maps are found (venngram's bundled maps, then any registered in the process) is experimental and still growing:
  a shared registry of maps is planned.